# Notebook 2: Clinical Risk Stratification Model
## XGBoost Risk Tier Prediction + Cost Forecasting

This notebook covers:
- Feature engineering from HCC flags and demographics
- XGBoost classifier (risk tier) and regressor (annual cost)
- Model evaluation: accuracy, MAE, R²
- Feature importance analysis
- Risk score distribution for ACO outreach prioritisation
- High-risk beneficiary identification


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import warnings; warnings.filterwarnings("ignore")

from src.data_generator import generate_beneficiary_cohort, generate_utilization_panel
from src.raf_calculator import calculate_raf_batch
from src.risk_stratification import engineer_features, train_and_evaluate, FEATURE_COLS

sns.set_style("whitegrid")
ACCENT = "#1B4F72"
PALETTE = ["#85C1E9", "#2E86C1", ACCENT]
print("Imports OK")


## 1. Generate Data & Engineer Features

In [ ]:
print("Generating cohort...")
cohort = generate_beneficiary_cohort(n=50_000)
panel  = generate_utilization_panel(cohort)
cohort_raf = calculate_raf_batch(cohort)

print("Engineering features...")
X = engineer_features(cohort_raf)
pre_costs = panel[panel["year"]==0].set_index("bene_id")["total_cost"]
X["actual_cost"] = X["bene_id"].map(pre_costs)
X = X.dropna(subset=["actual_cost"])

print(f"Feature matrix: {X.shape}")
print(f"\nRisk tier distribution:")
print(X["risk_tier"].value_counts().to_string())


## 2. Train Model & Evaluate

In [ ]:
results = train_and_evaluate(cohort_raf, panel)
model   = results["model"]
metrics = results["metrics"]
preds   = results["test_predictions"]
fi      = results["feature_importance"]
X_test  = results["X_test"]

print("\nFull Classification Report:")
print(classification_report(preds["actual_tier"], preds["predicted_tier"]))


## 3. Confusion Matrix

In [ ]:
tier_order = ["low", "moderate", "high"]
cm = confusion_matrix(preds["actual_tier"], preds["predicted_tier"], labels=tier_order)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=tier_order, yticklabels=tier_order, ax=ax)
ax.set_xlabel("Predicted Tier", fontsize=11)
ax.set_ylabel("Actual Tier", fontsize=11)
ax.set_title(f"Confusion Matrix — Risk Tier Classification\nAccuracy: {metrics['tier_accuracy']:.1%}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fi_top = fi.head(15)
colors = [ACCENT if i < 3 else "#2E86C1" if i < 8 else "#85C1E9"
          for i in range(len(fi_top))]
ax.barh(fi_top["feature"][::-1], fi_top["importance"][::-1],
        color=colors[::-1], edgecolor="white")
ax.set_xlabel("XGBoost Feature Importance (Gain)", fontsize=11)
ax.set_title("Top 15 Features — Risk Stratification Model", fontsize=12, fontweight="bold")
ax.axvline(fi_top["importance"].mean(), color="red", linestyle="--",
           linewidth=1, alpha=0.6, label="Mean importance")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 5. Predicted vs Actual Cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
sample = preds.sample(3000, random_state=42)
for tier, color in [("low","#85C1E9"),("moderate","#2E86C1"),("high",ACCENT)]:
    s = sample[sample["actual_tier"]==tier]
    axes[0].scatter(s["actual_cost"], s["predicted_cost"],
                    alpha=0.3, s=8, color=color, label=tier)
lim = max(sample["actual_cost"].max(), sample["predicted_cost"].max())
axes[0].plot([0,lim],[0,lim],"r--",linewidth=1.5,label="Perfect")
axes[0].set_xlabel("Actual Annual Cost ($)"); axes[0].set_ylabel("Predicted Annual Cost ($)")
axes[0].set_title(f"Predicted vs Actual Cost\nR²={metrics['cost_r2']:.3f}  MAE=${metrics['cost_mae']:,.0f}",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8)

# Error distribution
errors = preds["predicted_cost"] - preds["actual_cost"]
axes[1].hist(errors, bins=60, color=ACCENT, alpha=0.8, edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].axvline(errors.mean(), color="orange", linestyle="--",
                label=f"Mean error: ${errors.mean():,.0f}")
axes[1].set_xlabel("Prediction Error ($)"); axes[1].set_ylabel("Count")
axes[1].set_title("Prediction Error Distribution", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


## 6. High-Risk Beneficiary Identification

In an ACO setting, the risk model drives proactive outreach. Here we identify the top 10% highest-risk beneficiaries.

In [ ]:
# Score all beneficiaries
all_preds = model.predict(engineer_features(cohort_raf))
all_preds["bene_id"] = cohort_raf["bene_id"].values
all_preds["actual_tier"] = cohort_raf["risk_tier"].values
all_preds["age"] = cohort_raf["age"].values
all_preds["raf_score"] = cohort_raf["raf_score"].values

# Top 10% by predicted cost
threshold = all_preds["predicted_cost"].quantile(0.90)
high_risk = all_preds[all_preds["predicted_cost"] >= threshold].sort_values(
    "predicted_cost", ascending=False
)

print(f"High-risk cohort (top 10%): {len(high_risk):,} beneficiaries")
print(f"Predicted cost threshold:   ${threshold:,.0f}/year")
print(f"Mean predicted cost:        ${high_risk['predicted_cost'].mean():,.0f}/year")
print(f"Mean RAF score:             {high_risk['raf_score'].mean():.3f}")
print(f"\nRisk tier composition:")
print(high_risk["actual_tier"].value_counts().to_string())
print(f"\nTop 10 highest-risk beneficiaries:")
display(high_risk[["bene_id","age","raf_score","predicted_cost","predicted_tier"]].head(10).to_string(index=False))
